# Gold Set 생성 — Step 2: LLM Judge Relevance Labeling

`goldset_candidates.jsonl`의 후보 도서 각각에 대해 LLM judge가 relevance grade(0~3)를 부여하고, 최종 골드셋을 저장한다.

---

**입력**
- `evaluation/dataset/goldset_candidates.jsonl` — Step 1에서 생성한 후보 풀

**출력**
- `evaluation/dataset/goldset_all_scenarios.jsonl`
- `evaluation/dataset/goldset_all_scenarios.json`

**이전 단계**
- `01a_goldset_retrieval.ipynb` — BM25 / Dense / Hybrid 검색 후 후보 풀 저장

---

**판정 기준**
- `session_signals`를 최우선 기준으로 사용
- `onboarding_signals`는 충돌하지 않는 경우에만 보조 반영
- `retrieval_info`(rank/score/source)는 판정 입력에서 제외하되, 출력에는 유지
- grade 기준: 3=매우적합 / 2=적합 / 1=부분관련 / 0=부적합
- `binary_label = 1 if relevance_grade >= 2 else 0`

In [12]:
import os
os.chdir("/home/ubuntu/work/somin/backend")
os.getcwd()

'/home/ubuntu/work/somin/backend'

In [13]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.core.config import settings
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

In [14]:
CLOVA_API_KEY = settings.CLOVA_API_KEY
URL           = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"

REPO_ROOT       = Path("/home/ubuntu/work/somin")
DATASET_DIR     = REPO_ROOT / "evaluation" / "dataset"
CANDIDATES_PATH = DATASET_DIR / "goldset_candidates.jsonl"
JSONL_PATH      = DATASET_DIR / "goldset_all_scenarios3.jsonl"
FINAL_JSON_PATH = DATASET_DIR / "goldset_all_scenarios3.json"
# 정현:goldset_all_scenarios1.jsonl
# 다현:goldset_all_scenarios2.jsonl
# 소민:goldset_all_scenarios3.jsonl

CONCURRENCY = 8    # 동시 요청 수 (429 발생 시 5로 낮춤)
MAX_RETRIES = 5
LIMIT       = None  # 테스트 시 숫자로 변경, 전체 실행 시 None

candidates = []
with CANDIDATES_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        candidates.append(json.loads(line))

print(f"후보 도서 수: {len(candidates):,}")
print(f"동시 요청 수: {CONCURRENCY}")
print(f"LIMIT: {LIMIT}")

from collections import Counter
query_counts = Counter(c["query_id"] for c in candidates)
for qid in sorted(query_counts):
    print(f"  {qid}: {query_counts[qid]}")

후보 도서 수: 2,543
동시 요청 수: 8
LIMIT: None
  S01: 115
  S02: 115
  S03: 85
  S04: 102
  S05: 130
  S06: 120
  S07: 67
  S08: 125
  S09: 137
  S10: 116
  S11: 141
  S12: 125
  S13: 137
  S14: 130
  S15: 114
  S16: 145
  S17: 116
  S18: 122
  S19: 136
  S20: 134
  S21: 131


In [ ]:
SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청
- semantic_query: 사용자 의도를 의미적으로 표현한 쿼리
- keyword_query: 핵심 키워드 목록
- filters: 반드시 충족해야 하는 카테고리 조건
    - cate_depth1: 카테고리 조건
    - custom_constraints: 주제/테마 조건
- constraints: 반드시 충족해야 하는 수치 조건 (페이지 수, 출판연도 등)
- score_boost: 충족하면 좋은 선호 카테고리/주제
- session_signals: 현재 대화에서 파악한 사용자 의도
- onboarding_signals: 사용자 장기 취향 정보

2. book: 평가할 도서 정보
- title, author: 도서 기본 정보
- publish_date: 출판일 (YYYY-MM-DD) → constraints.pub_year 판단에 사용
- page: 페이지 수 → constraints.page_range 판단에 사용
- large_cate: 대분류 → filters.cate_depth1 판단에 사용
- mid_cate: 중분류 → score_boost.cate_depth2 판단에 사용
- small_cate: 소분류/장르 → score_boost 판단에 사용
- book_intro: 도서 소개 → 핵심 의도 부합 여부 판단에 사용
- book_index: 목차 → 주제 파악 보조
- review_text: 독자 리뷰 → 분위기/독서 경험 파악 보조

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- session_signals를 최우선 기준으로 판단하세요.
- onboarding_signals는 session_signals와 충돌하지 않는 경우에만 보조적으로 반영하세요.
- session_signals와 onboarding_signals가 충돌하면 session_signals를 우선하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.
- retrieval rank, score, source 정보는 판단에 사용하지 마세요.
- 반드시 JSON만 출력하세요. 설명 문장, markdown, 코드블록은 출력하지 마세요.

[Grade 기준]
3 = 매우 적합. 핵심 의도 + 대부분의 조건에 잘 부합.
2 = 적합. 핵심 의도는 맞지만 일부 조건이 약하거나 빠짐.
1 = 부분 관련. 주제는 겹치나 분위기/대상/장르가 실제로 다름.
    단순 키워드 일치만 있고 독서 경험이 맞지 않는 경우.
0 = 부적합. 핵심 의도와 맞지 않거나 filters/constraints 미충족.

출력 형식:
{
  "relevance_grade": 0 또는 1 또는 2 또는 3,
  "reason": "판정 근거를 2-3문장으로 설명"
}
"""

EXCLUDE_KEYS = {
    # retrieval 메타데이터
    "retrieval_info", "rank", "score", "source",
    "retrieval_source", "retrieval_rank", "retrieval_score",
    "bm25_score", "dense_score", "hybrid_score",
    "main_score", "onboarding_score",
    "disliked_penalty", "disliked_similarity",

    # 파이프라인 메타데이터
    "query_id", "rag_query",

    # 쿼리 재구성 단계에서 이미 소비됨
    "anchor", "session_signals"

    # 기존 평가 필드: 재실행 시 LLM 편향 방지
    "relevance_grade", "binary_label", "label_status",
    "llm_raw_output", "llm_error",
}

In [ ]:

def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }


def make_book_for_judge(book: dict) -> dict:
    """LLM judge 입력에서 retrieval 정보 및 파이프라인 메타데이터를 제거한다."""
    return {
        k: v for k, v in book.items()
        if k not in EXCLUDE_KEYS and not str(k).startswith("retrieval_")
    }


def parse_hcx_response(text: str) -> str:
    if "event:" in text:
        last_data = None
        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = data_text = None
            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()
            if event_type == "result" and data_text:
                last_data = data_text
        if last_data is not None:
            return json.loads(last_data)["message"]["content"]

    try:
        data = json.loads(text)
        if "result" in data:
            return data["result"]["message"]["content"]
        if "message" in data:
            return data["message"]["content"]
    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text: str):
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))
        if grade is None:
            return None, "parse_failed", "missing relevance_grade"
        grade = int(grade)
        if grade not in [0, 1, 2, 3]:
            return None, "parse_failed", f"invalid relevance_grade: {grade}"
        return grade, "success", None
    except Exception as e:
        return None, "parse_failed", str(e)


def grade_to_binary(grade):
    if grade is None:
        return None
    return 1 if grade >= 2 else 0


def blocking_label_one_book(request_data: dict, book: dict) -> dict:
    """동기 HTTP 요청 함수. asyncio.to_thread()로 thread pool에서 호출된다."""
    query_id    = request_data.get("query_id", "unknown")
    user_prompt = json.dumps({"request": request_data, "book": make_book_for_judge(book)},
                             ensure_ascii=False, indent=2)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        "topP": 0.1, "topK": 0, "max_tokens": 100,
        "temperature": 0.0, "repetitionPenalty": 1.0,
        "includeAiFilters": False, "seed": 42,
    }

    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            res = requests.post(URL, headers=make_headers(), json=body, timeout=60)
            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                grade, label_status, parse_error = extract_grade(llm_text)
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": grade,
                    "binary_label":    grade_to_binary(grade),
                    "label_status":    label_status,
                    "llm_raw_output":  llm_text,
                    "llm_error":       parse_error,
                }
            last_error = f"{res.status_code} / {res.text[:500]}"
            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)
            return {
                **book,
                "query_id":        query_id,
                "relevance_grade": None, "binary_label": None,
                "label_status":    "api_failed",
                "llm_raw_output":  None, "llm_error": last_error,
            }
        except Exception as e:
            last_error = str(e)
            if attempt == MAX_RETRIES - 1:
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": None, "binary_label": None,
                    "label_status":    "api_failed",
                    "llm_raw_output":  None, "llm_error": last_error,
                }
            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            time.sleep(wait)


async def label_one_book_async(request_data, book, semaphore, write_lock, jsonl_file, pbar):
    """
    비동기 래퍼:
    - semaphore로 동시 요청 수 제한
    - asyncio.to_thread로 blocking HTTP 호출을 thread pool에서 실행
    - write_lock으로 JSONL 파일 쓰기 직렬화
    """
    async with semaphore:
        try:
            result = await asyncio.to_thread(blocking_label_one_book, request_data, book)
        except Exception as e:
            result = {
                **book,
                "query_id":        request_data.get("query_id", "unknown"),
                "relevance_grade": None, "binary_label": None,
                "label_status":    "api_failed",
                "llm_raw_output":  None, "llm_error": str(e),
            }

    async with write_lock:
        jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        jsonl_file.flush()

    pbar.update(1)
    return result

# LLM-judge 실행

In [16]:
async def main():
    # resume: 이미 처리된 (query_id, isbn) 스킵
    processed_keys: set = set()
    already_labeled: list = []

    if JSONL_PATH.exists():
        with JSONL_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    item = json.loads(line)
                    key  = (item.get("query_id"), item.get("isbn"))
                    if key not in processed_keys:
                        processed_keys.add(key)
                        already_labeled.append(item)
                except json.JSONDecodeError:
                    continue
        print(f"이미 처리된 도서 수: {len(processed_keys):,}")

    # 01a에서 저장한 rag_query를 request_data로 사용
    to_process = [
        book for book in candidates
        if (book.get("query_id"), book.get("isbn")) not in processed_keys
        and book.get("rag_query")
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"전체 후보: {len(candidates):,}  |  처리 대상: {len(to_process):,}")

    if not to_process:
        print("처리할 항목이 없습니다.")
        return already_labeled

    semaphore  = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar       = tqdm(total=len(to_process), desc="라벨링 중", unit="book")

    with JSONL_PATH.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            label_one_book_async(
                book["rag_query"],
                book,
                semaphore,
                write_lock,
                jsonl_file,
                pbar,
            )
            for book in to_process
        ]
        new_results = await asyncio.gather(*tasks)

    pbar.close()

    # 원본 후보 순서 기준 정렬 후 JSON 저장
    all_results = already_labeled + list(new_results)
    order = {(e.get("query_id"), e.get("isbn")): i for i, e in enumerate(candidates)}
    all_results.sort(key=lambda x: order.get((x.get("query_id"), x.get("isbn")), 99999))

    with FINAL_JSON_PATH.open("w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    total_success = sum(1 for r in all_results if r.get("label_status") == "success")
    total_failed  = sum(1 for r in all_results if r.get("label_status") != "success")
    print(f"\n총 항목: {len(all_results):,}  성공: {total_success:,}  실패: {total_failed:,}")
    print(f"JSONL 저장 완료: {JSONL_PATH}")
    print(f"JSON  저장 완료: {FINAL_JSON_PATH}")

    return all_results


results = await main()

전체 후보: 2,543  |  처리 대상: 2,543


라벨링 중: 100%|██████████| 2543/2543 [1:22:11<00:00,  1.94s/book]  



총 항목: 2,543  성공: 1,795  실패: 748
JSONL 저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_all_scenarios3.jsonl
JSON  저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_all_scenarios3.json


# api_failed 재시도

`main()` 완료 후 `label_status == "api_failed"` / `"parse_failed"` 항목만 골라 재호출한다.
성공한 항목은 그대로 유지하고, 결과는 FINAL_JSON_PATH에 덮어쓴다.

In [24]:
async def retry_api_failed():
    """api_failed + parse_failed 항목을 FINAL_JSON_PATH에서 읽어 재호출 후 덮어쓴다."""
    if not FINAL_JSON_PATH.exists():
        print(f"파일 없음: {FINAL_JSON_PATH}\nmain()을 먼저 실행하세요.")
        return []

    with FINAL_JSON_PATH.open("r", encoding="utf-8") as f:
        all_results = json.load(f)

    to_retry = [
        e for e in all_results
        if e.get("label_status") in ("api_failed", "parse_failed")
        or e.get("relevance_grade") is None
    ]
    print(f"전체: {len(all_results):,}개  |  재시도 대상: {len(to_retry):,}개")

    if not to_retry:
        print("재시도할 항목이 없습니다.")
        return all_results

    semaphore = asyncio.Semaphore(CONCURRENCY)
    pbar      = tqdm(total=len(to_retry), desc="재시도 중", unit="book")

    async def retry_one(entry):
        rag_query = entry.get("rag_query")
        if not rag_query:
            pbar.update(1)
            return entry
        async with semaphore:
            try:
                result = await asyncio.to_thread(blocking_label_one_book, rag_query, entry)
            except Exception as e:
                result = {**entry, "label_status": "api_failed", "llm_error": str(e)}
        pbar.update(1)
        return result

    new_results = await asyncio.gather(*[retry_one(e) for e in to_retry])
    pbar.close()

    key_to_new = {(r.get("query_id"), r.get("isbn")): r for r in new_results}
    updated = [
        key_to_new.get((e.get("query_id"), e.get("isbn")), e)
        for e in all_results
    ]

    with FINAL_JSON_PATH.open("w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    now_success  = sum(1 for r in new_results if r.get("label_status") == "success")
    still_failed = sum(1 for r in new_results if r.get("label_status") in ("api_failed", "parse_failed"))
    print(f"\n재시도 결과 ({len(to_retry)}개)  성공: {now_success:,}  여전히 실패: {still_failed:,}")
    print(f"JSON 저장 완료: {FINAL_JSON_PATH}")
    return updated


results = await retry_api_failed()

전체: 2,543개  |  재시도 대상: 10개


재시도 중:   0%|          | 0/10 [00:00<?, ?book/s]

재시도 중: 100%|██████████| 10/10 [00:14<00:00,  1.49s/book]



재시도 결과 (10개)  성공: 10  여전히 실패: 0
JSON 저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_all_scenarios3.json


# 결과 확인

In [35]:
# rows = []
# with JSONL_PATH.open("r", encoding="utf-8") as f:
#     for line in f:
#         if line.strip():
#             rows.append(json.loads(line))

with FINAL_JSON_PATH.open("r", encoding="utf-8") as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
print(f"전체 라벨링 결과 수: {len(df):,}")
df.head(2)

전체 라벨링 결과 수: 2,543


,isbn,title,author,publisher,publish_date,page,large_cate,mid_cate,small_cate,book_intro,...,review_count,review_text,retrieval_info,query_id,rag_query,relevance_grade,binary_label,label_status,llm_raw_output,llm_error
0,9791157063598,철학자와 하녀 - 하루하루를 살아가는 마이너리티의 철학,고병권 저,메디치미디어,2024-07-03,256,[인문],[철학],[철학에세이],현실을 바꿔나갈 힘을 얻는 ‘현장의 인문학’\n이 책의 제목에 나오는 ‘하녀’는 권...,...,12,"철학책을 재미있게 읽을 수 있으며, 삶의 방식과 철학적 사고를 연결해주는 유익한 경...","[{'source': 'bm25_full', 'rank': 1, 'score': 6...",S01,"{'keyword_query': ['실존주의적 주제', '철학적 성찰', '인문학 ...",2,1,success,"{\n ""relevance_grade"": 2,\n ""reason"": ""책은 철학...",None
1,9791165347772,그렇게 나를 만들어간다,장마리아 저,쌤앤파커스,2023-08-09,216,"[시/에세이, 예술/대중문화]","[나라별 에세이, 미술, 테마에세이]","[교양미술, 그림에세이, 한국에세이]","“오늘도 맑았다가 흐렸다, 그렇게 미치도록 쨍하기를!”\n미술계와 셀럽, 젊은 예술...",...,2,자신을 찾아가는 과정을 담담하게 풀어내며 공감을 주는 에세이입니다. 자기 자신을 알...,"[{'source': 'bm25_full', 'rank': 2, 'score': 5...",S01,"{'keyword_query': ['실존주의적 주제', '철학적 성찰', '인문학 ...",1,0,success,"{\n ""relevance_grade"": 1,\n ""reason"": ""책은 자기...",None


In [36]:
status_dist = df["label_status"].value_counts(dropna=False).reset_index()
status_dist.columns = ["label_status", "count"]
status_dist["ratio"] = status_dist["count"] / len(df)
status_dist

,label_status,count,ratio
0,success,2543,1.0


In [37]:
success_df = df[df["label_status"] == "success"].copy()
print(f"성공 라벨 수: {len(success_df):,}")
print(f"실패/제외 수: {len(df) - len(success_df):,}")

성공 라벨 수: 2,543
실패/제외 수: 0


In [38]:
grade_dist = (
    success_df["relevance_grade"]
    .value_counts()
    .sort_index()
    .reset_index()
)
grade_dist.columns = ["relevance_grade", "count"]
grade_dist["ratio"] = grade_dist["count"] / len(success_df)
grade_dist

,relevance_grade,count,ratio
0,0,274,0.107747
1,1,1601,0.629571
2,2,468,0.184035
3,3,200,0.078647


In [39]:
from collections import defaultdict
stats = defaultdict(lambda: defaultdict(int))
for item in results:
    stats[item.get("query_id", "unknown")][item.get("relevance_grade")] += 1

print(f"{'query_id':>10}  {'grade 0':>7}  {'grade 1':>7}  {'grade 2':>7}  {'grade 3':>7}  {'total':>7}")
print("-" * 55)
for qid in sorted(stats):
    g = stats[qid]
    total = sum(g.values())
    print(f"{qid:>10}  {g[0]:>7}  {g[1]:>7}  {g[2]:>7}  {g[3]:>7}  {total:>7}")

  query_id  grade 0  grade 1  grade 2  grade 3    total
-------------------------------------------------------
       S01        9       87       13        6      115
       S02       14       93        6        2      115
       S03       26       54        4        1       85
       S04        6       78       14        4      102
       S05       15      100       12        3      130
       S06        0       57       55        8      120
       S07       33       24        8        2       67
       S08       19       82       21        3      125
       S09       14      103       17        3      137
       S10        3       62       41       10      116
       S11       22      106       12        1      141
       S12       10       49       11       55      125
       S13       16       91       27        3      137
       S14        5      109       12        4      130
       S15       17       93        4        0      114
       S16       25       95       20        5  